In [ ]:
import logging

logging.basicConfig(level="DEBUG")
logging.getLogger("fsspec").setLevel(logging.WARNING)
logging.getLogger("matplotlib").setLevel(logging.WARNING)

In [ ]:
import torch
from darts.utils.missing_values import extract_subseries
from darts.models import LinearRegressionModel
from darts.dataprocessing.transformers import Scaler
from sklearn.preprocessing import StandardScaler

import darts
from aare.constants import TEMP

import pandas as pd
from aare.AareDataset import AareDataset

from aare.evaluation.evaluation import evaluate_model
from aare.params import read_params
from aare.preparation import (
    prepare_ts_aare_temp,
    resample,
    interpolate_aare_temp,
)
from aare.remote_existenz_store import RemoteExistenzStore
from aare.utils import to_ts

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (16, 9)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
torch.set_float32_matmul_precision("medium")

In [ ]:
params = read_params()
store = RemoteExistenzStore()
ds = AareDataset.from_conf()

# Linear Regression

Even though we know that there are non-linear relationships at play, a simple linear regression model using the last water temperature, the relevant air temperature lag(s), as well as the time, might give a good comparison to other approaches.

In [ ]:
validation_params = params["validation"]
stride = validation_params["stride"]
min_lookback_hours = validation_params["min_lookback_hours"]
forecast_horizon = params["general"]["forecast_horizon"]

In [ ]:
train = prepare_ts_aare_temp(ds.get_train())
train

In [ ]:
val = prepare_ts_aare_temp(ds.get_val())
val

## TODO Refactor fetching other features

Need some sort of standardized method to fetch and prepare certain features up until they're combined into TimeSeries instances split into train and val, all without any gaps.

In [ ]:
tt_bern = store.query((params["split"]["train_split"], params["split"]["test_split"]), "smn/tt:mean_1h@bern")
tt_bern

In [ ]:
tt_bern = resample(tt_bern)
tt_bern = interpolate_aare_temp(tt_bern, drop_filled=True, columns="tt_bern")
tt_bern = to_ts(tt_bern, col="tt_bern")
tt_bern

In [ ]:
tt_bern_train, tt_bern_val = tt_bern.split_after(pd.Timestamp(params["split"]["val_split"]))
(len(tt_bern_train), len(tt_bern_val))

In [ ]:
# we lose a lot of data because tt is only available from 2013
train_combined = darts.concatenate([train.slice_intersect(tt_bern_train), tt_bern_train], axis="component")
val_combined = darts.concatenate([val.slice_intersect(tt_bern_val), tt_bern_val], axis="component")

In [ ]:
train_combined

In [ ]:
train_combined_subs = extract_subseries(train_combined, mode="any")
val_combined_subs = extract_subseries(val_combined, mode="any")

In [ ]:
train_target_subs = [ts[TEMP] for ts in train_combined_subs]
train_fc_subs = [ts[["tt_bern"]] for ts in train_combined_subs]
val_target_subs = [ts[TEMP] for ts in val_combined_subs]
val_fc_subs = [ts[["tt_bern"]] for ts in val_combined_subs]

In [ ]:
scaler_target = Scaler(StandardScaler(), global_fit=True)
scaler_fc = Scaler(StandardScaler(), global_fit=True)

In [ ]:
scaler_target.fit(train_target_subs)
scaler_fc.fit(train_fc_subs)

In [ ]:
from aare.compat.types import DataTransformers

data_transformers: DataTransformers = {
    "series": scaler_target,
    "future_covariates": scaler_fc,
}

In [ ]:
model = LinearRegressionModel(
    lags=1,  # use water temp at last hour
    lags_future_covariates=[-1],  # use air temp at last hour (highest corr)
    output_chunk_length=1,  # only predict 1 hour into the future
    # likelihood="quantile",
    quantiles=[0.25, 0.5, 0.75],
    random_state=42,
    multi_models=True,  # 1 model for each output step (doesn't matter if out_chunk_length is 1 anyway)
    use_static_covariates=False,  # currently no static covariates
)

In [ ]:
model.fit(
    series=scaler_target.transform(train_target_subs),
    future_covariates=scaler_fc.transform(train_fc_subs),
)

In [ ]:
ex_val_target = val_target_subs[-1][29:30]
ex_val_fc = val_fc_subs[-1][: 30 + forecast_horizon]
ex_pred = model.predict(
    n=forecast_horizon,
    # num_samples=128,
    series=scaler_target.transform(ex_val_target),
    future_covariates=scaler_fc.transform(ex_val_fc),
)
ex_pred = scaler_target.inverse_transform(ex_pred)
ex_pred

In [ ]:
val_target_subs[-1][: 30 + forecast_horizon].plot(label="actual")
ex_val_fc.plot(label="air temp")
ex_pred.plot(label="prediction")

In [ ]:
metrics, samples = evaluate_model(
    model,
    val_target_subs,
    forecast_horizon,
    stride,
    min_lookback_hours,
    future_cov=val_fc_subs,
    # num_samples=128,
    data_transformers=data_transformers,
)

In [ ]:
metrics

In [ ]:
_ = samples.plot("LR", with_covariates=True)

In [ ]:
from aare.utils import DATA_FOLDER

assert False
# just in case. for actual experimentation a better setup should be used
model.save(DATA_FOLDER / "models/initial-lr.pkl")

# Conclusion from initial experiment

1. Training without quantiles (no probabilistic forecasting) is really quick, but it took 1h20m to train it with .25, .50 and .75 quantiles. So it's unbearably slow by not using the GPU and also not multi-threading I think.
1. I think this might be a good approach to model the process. It can follow the patterns quite well, but the overall MAE and RMSE are TERRIBLE because it seems to give too much weight to non-standard changes in air temperature. This might be ~~because of the missing normalization but also probably~~ because it can only model linear relationships.

Next steps:
- Create a better setup to train and evaluate this model, especially in regard to loading and preparing covariates
- Also fix the train/val split given we will now only use data from 2013 onwards
- DONE Normalize the data before training :)
- Try adding non-linear transformations of the air temperature
- Add other features, namely sunshine duration, flow and maybe time-encoding
- Try more complex models (prob MLP based like TSMixer), these would also use the GPU and are torch based

When iterating on this, save time by not using probabilistic forecasting at first.